# Tutorial 2: Feature Engineering for Pressing Analysis

This tutorial demonstrates how to extract pressing features from build-up tracking data. You'll learn:

- How coordinate normalization works
- How ball carrier inference enables pressure metrics
- How to compute pressing intensity and team compactness features
- How to interpret feature values tactically

## Prerequisites

Complete Tutorial 1 to extract build-ups into `data/processed/rm_pressing_tutorial/`

## Setup

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
import json

# Feature engineering modules
from src.features.feature_engineering import extract_features_for_build_up
from src.features.services.normalization import normalize_coordinates
from src.features.services.possession import infer_ball_carrier
from src.features.services.pressure import aggregate_pressure_features
from src.features.services.compactness import aggregate_compactness_features
from src.features.services.window_loader import WindowLoader
from src.features.services.utils import prepare_frame_data, time_to_seconds
from src.features.services.metadata import enrich_with_team_id

# Configuration
PROCESSED_ROOT = Path("data/processed/rm_pressing_tutorial")

sns.set_style("whitegrid")

## Step 1: Load Build-Up Data

Use `WindowLoader` to access extracted build-ups.

In [ ]:
loader = WindowLoader(PROCESSED_ROOT)
index = loader.index

print(f"Loaded {len(index)} build-ups")
print(f"\nIndex columns: {index.columns.tolist()}")

# Load first build-up
build_up_id = index.iloc[0]['build_up_id']
df = loader.load_build_up(build_up_id)
metadata = loader.get_metadata(build_up_id)

print(f"\nBuild-up {build_up_id}:")
print(f"  {len(df)} frames")
print(f"  Columns: {df.columns.tolist()[:10]}...")  # First 10

## Step 2: Coordinate Normalization

Normalize coordinates so opponent always attacks towards the right (+X direction). This enables comparison across build-ups regardless of which side the goalkeeper started on.

In [ ]:
# Prepare data
df = prepare_frame_data(df)
df = enrich_with_team_id(df, metadata['game_id'])

# Show original coordinates
print(f"GK Side: {metadata.get('gk_side', 'unknown')}")
print(f"\nOriginal coordinates (first frame, 3 players):")
sample = df[df['frame'] == df['frame'].min()].head(3)
print(sample[['player_id', 'x', 'y']])

# Normalize
df_norm = normalize_coordinates(df, metadata.get('gk_side', 'left'))

print(f"\nNormalized coordinates (same players):")
sample_norm = df_norm[df_norm['frame'] == df_norm['frame'].min()].head(3)
print(sample_norm[['player_id', 'x_norm', 'y_norm']])

# Visualize transformation
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Original
first_frame = df[df['frame'] == df['frame'].min()]
ax1.scatter(first_frame['x'], first_frame['y'], alpha=0.6)
ax1.set_xlim(-52.5, 52.5)
ax1.set_ylim(-34, 34)
ax1.set_xlabel('X (meters)')
ax1.set_ylabel('Y (meters)')
ax1.set_title('Original Coordinates')
ax1.grid(alpha=0.3)

# Normalized
first_frame_norm = df_norm[df_norm['frame'] == df_norm['frame'].min()]
ax2.scatter(first_frame_norm['x_norm'], first_frame_norm['y_norm'], alpha=0.6, color='orange')
ax2.set_xlim(-52.5, 52.5)
ax2.set_ylim(-34, 34)
ax2.set_xlabel('X (meters)')
ax2.set_ylabel('Y (meters)')
ax2.set_title('Normalized Coordinates (Opponent attacks →)')
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## Step 3: Ball Carrier Inference

Infer which opponent player has possession at each frame. This is critical for pressure metrics.

In [ ]:
# Infer ball carrier
df_norm = infer_ball_carrier(df_norm, metadata.get('opponent_team_id'))

# Check results
carrier_counts = df_norm.dropna(subset=['ball_carrier_id']).groupby('ball_carrier_id').size()
print(f"Ball carriers identified: {len(carrier_counts)} players")
print(f"\nTop 3 ball carriers (most frames):")
print(carrier_counts.sort_values(ascending=False).head(3))

# Visualize carrier over time
df_norm['time_seconds'] = df_norm['time'].apply(time_to_seconds)
carrier_frames = df_norm.dropna(subset=['ball_carrier_id'])

plt.figure(figsize=(12, 4))
for player_id in carrier_counts.index[:5]:  # Top 5 carriers
    player_frames = carrier_frames[carrier_frames['ball_carrier_id'] == player_id]
    plt.scatter(player_frames['time_seconds'], [player_id]*len(player_frames), 
                alpha=0.6, s=20, label=f'Player {player_id}')

kick_time = time_to_seconds(str(metadata.get('kick_time')))
plt.axvline(kick_time, color='red', linestyle='--', label='Kick time')
plt.xlabel('Time (seconds)')
plt.ylabel('Player ID')
plt.title('Ball Carrier Over Time')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Step 4: Pressure Features

Compute 7 pressing intensity metrics:
1. Time to first pressure
2. Pressure frames ratio
3. Number of pressure bursts
4. Max closing velocity
5. Mean presser-carrier distance
6. Min presser-carrier distance
7. Closest presser ID

In [ ]:
from src.features.config import FeatureConfig

config = FeatureConfig()
pressure_feats = aggregate_pressure_features(
    df_norm, 
    metadata.get('rm_team_id'),
    kick_time,
    config
)

print("Pressure Features:")
for key, val in pressure_feats.items():
    print(f"  {key}: {val:.3f}" if isinstance(val, (int, float)) else f"  {key}: {val}")

# Interpret values
print("\nInterpretation:")
if pressure_feats['t_first_pressure_s'] < 2:
    print("  ⚡ AGGRESSIVE: First pressure within 2s (high press)")
elif pressure_feats['t_first_pressure_s'] < 4:
    print("  ⚖️  CONTROLLED: First pressure 2-4s (mid-block)")
else:
    print("  🛡️ REACTIVE: First pressure >4s (low block)")

if pressure_feats['pressure_frames_ratio'] > 0.5:
    print("  💪 SUSTAINED: High pressure density (>50% of frames)")
elif pressure_feats['pressure_frames_ratio'] > 0.3:
    print("  📊 MODERATE: Moderate pressure density (30-50%)")
else:
    print("  🌊 SPORADIC: Low pressure density (<30%)")

## Step 5: Compactness Features

Compute 6 team shape metrics:
1. Mean width (lateral spread)
2. Mean length (longitudinal spread)
3. Mean convex hull area
4. Line height (defensive line position)
5. Mean centroid X
6. Mean centroid Y

In [ ]:
compactness_feats = aggregate_compactness_features(
    df_norm,
    metadata.get('rm_team_id'),
    config
)

print("Compactness Features:")
for key, val in compactness_feats.items():
    print(f"  {key}: {val:.2f}")

# Interpret values
print("\nInterpretation:")
if compactness_feats['rm_width_mean_m'] < 20:
    print("  📏 COMPACT: Narrow width (<20m) - coordinated press")
elif compactness_feats['rm_width_mean_m'] < 30:
    print("  📏 BALANCED: Moderate width (20-30m)")
else:
    print("  📏 EXTENDED: Wide shape (>30m) - potential gaps")

if compactness_feats['rm_line_height_median_x_mean'] > 10:
    print("  ⬆️  HIGH LINE: Defensive line >10m (aggressive)")
elif compactness_feats['rm_line_height_median_x_mean'] > -10:
    print("  ↔️  MID BLOCK: Defensive line -10 to +10m")
else:
    print("  ⬇️  DEEP BLOCK: Defensive line <-10m (defensive)")

## Step 6: Visualize Team Shape Evolution

Plot how Real Madrid's compactness changes over the build-up window.

In [ ]:
from src.features.services.compactness import compute_compactness_metrics_per_frame

# Compute per-frame metrics
frame_metrics = []
for frame_id in df_norm['frame'].unique():
    frame_df = df_norm[df_norm['frame'] == frame_id]
    t = frame_df['time_seconds'].iloc[0]
    
    metrics = compute_compactness_metrics_per_frame(
        frame_df, 
        metadata.get('rm_team_id'), 
        config
    )
    
    if metrics:
        frame_metrics.append({'time': t, **metrics})

fm_df = pd.DataFrame(frame_metrics)

# Plot evolution
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

# Width
axes[0, 0].plot(fm_df['time'], fm_df['width_m'], color='blue', linewidth=2)
axes[0, 0].axvline(kick_time, color='red', linestyle='--', alpha=0.5)
axes[0, 0].set_ylabel('Width (m)')
axes[0, 0].set_title('Team Width Over Time')
axes[0, 0].grid(alpha=0.3)

# Length
axes[0, 1].plot(fm_df['time'], fm_df['length_m'], color='green', linewidth=2)
axes[0, 1].axvline(kick_time, color='red', linestyle='--', alpha=0.5)
axes[0, 1].set_ylabel('Length (m)')
axes[0, 1].set_title('Team Length Over Time')
axes[0, 1].grid(alpha=0.3)

# Hull Area
axes[1, 0].plot(fm_df['time'], fm_df['hull_area_m2'], color='purple', linewidth=2)
axes[1, 0].axvline(kick_time, color='red', linestyle='--', alpha=0.5, label='Kick')
axes[1, 0].set_xlabel('Time (s)')
axes[1, 0].set_ylabel('Hull Area (m²)')
axes[1, 0].set_title('Convex Hull Area Over Time')
axes[1, 0].legend()
axes[1, 0].grid(alpha=0.3)

# Line Height
axes[1, 1].plot(fm_df['time'], fm_df['line_height_median_x'], color='orange', linewidth=2)
axes[1, 1].axvline(kick_time, color='red', linestyle='--', alpha=0.5)
axes[1, 1].axhline(0, color='black', linestyle='-', alpha=0.2)
axes[1, 1].set_xlabel('Time (s)')
axes[1, 1].set_ylabel('Line Height (m)')
axes[1, 1].set_title('Defensive Line Height Over Time')
axes[1, 1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## Step 7: Extract All Features

Use the main pipeline to extract all 20+ features.

In [ ]:
all_features = extract_features_for_build_up(build_up_id, PROCESSED_ROOT, config)

print(f"Extracted {len(all_features)} features:")
print("\n" + "="*60)
for key, val in all_features.items():
    if isinstance(val, (int, float)):
        print(f"{key:40s}: {val:10.3f}")
    else:
        print(f"{key:40s}: {val}")
print("="*60)

## Step 8: Batch Feature Extraction

Extract features for all build-ups and analyze distributions.

In [ ]:
from tqdm.notebook import tqdm

# Extract features for all build-ups
all_build_up_features = []
for bid in tqdm(index['build_up_id'].tolist(), desc="Extracting features"):
    try:
        feats = extract_features_for_build_up(bid, PROCESSED_ROOT, config)
        feats['build_up_id'] = bid
        all_build_up_features.append(feats)
    except Exception as e:
        print(f"Error on {bid}: {e}")

features_df = pd.DataFrame(all_build_up_features)
print(f"\nExtracted features for {len(features_df)} build-ups")

# Save
features_df.to_parquet(PROCESSED_ROOT / "features.parquet")
print(f"Saved to {PROCESSED_ROOT / 'features.parquet'}")

## Step 9: Feature Distributions

Visualize distributions of key pressing metrics.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

# Time to first pressure
axes[0, 0].hist(features_df['t_first_pressure_s'].dropna(), bins=20, color='steelblue', edgecolor='black')
axes[0, 0].axvline(2, color='orange', linestyle='--', label='Aggressive (<2s)')
axes[0, 0].axvline(4, color='red', linestyle='--', label='Reactive (>4s)')
axes[0, 0].set_xlabel('Time to First Pressure (s)')
axes[0, 0].set_ylabel('Count')
axes[0, 0].set_title('Pressing Trigger Speed')
axes[0, 0].legend()

# Pressure frames ratio
axes[0, 1].hist(features_df['pressure_frames_ratio'].dropna(), bins=20, color='green', edgecolor='black')
axes[0, 1].axvline(0.3, color='orange', linestyle='--', alpha=0.7)
axes[0, 1].axvline(0.5, color='red', linestyle='--', alpha=0.7)
axes[0, 1].set_xlabel('Pressure Frames Ratio')
axes[0, 1].set_ylabel('Count')
axes[0, 1].set_title('Pressure Density')

# Team width
axes[0, 2].hist(features_df['rm_width_mean_m'].dropna(), bins=20, color='purple', edgecolor='black')
axes[0, 2].axvline(20, color='orange', linestyle='--', label='Compact (<20m)')
axes[0, 2].axvline(30, color='red', linestyle='--', label='Extended (>30m)')
axes[0, 2].set_xlabel('Team Width (m)')
axes[0, 2].set_ylabel('Count')
axes[0, 2].set_title('Lateral Compactness')
axes[0, 2].legend()

# Line height
axes[1, 0].hist(features_df['rm_line_height_median_x_mean'].dropna(), bins=20, color='darkred', edgecolor='black')
axes[1, 0].axvline(-10, color='blue', linestyle='--', label='Deep (<-10m)')
axes[1, 0].axvline(10, color='red', linestyle='--', label='High (>10m)')
axes[1, 0].set_xlabel('Line Height (m)')
axes[1, 0].set_ylabel('Count')
axes[1, 0].set_title('Defensive Line Position')
axes[1, 0].legend()

# Hull area
axes[1, 1].hist(features_df['rm_hull_area_mean_m2'].dropna(), bins=20, color='teal', edgecolor='black')
axes[1, 1].set_xlabel('Hull Area (m²)')
axes[1, 1].set_ylabel('Count')
axes[1, 1].set_title('Team Footprint')

# Pressure bursts
axes[1, 2].hist(features_df['n_pressure_bursts'].dropna(), bins=range(0, 10), color='coral', edgecolor='black')
axes[1, 2].set_xlabel('Number of Pressure Bursts')
axes[1, 2].set_ylabel('Count')
axes[1, 2].set_title('Pressing Episodes')

plt.tight_layout()
plt.show()

## Step 10: Correlation Analysis

Explore relationships between pressing and compactness features.

In [ ]:
# Select key features
key_features = [
    't_first_pressure_s',
    'pressure_frames_ratio',
    'n_pressure_bursts',
    'rm_width_mean_m',
    'rm_length_mean_m',
    'rm_hull_area_mean_m2',
    'rm_line_height_median_x_mean'
]

corr_matrix = features_df[key_features].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            square=True, linewidths=1, cbar_kws={"shrink": 0.8})
plt.title('Feature Correlation Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("\nKey Insights:")
print(f"  - Width vs Hull Area: r = {corr_matrix.loc['rm_width_mean_m', 'rm_hull_area_mean_m2']:.2f}")
print(f"  - Pressure Time vs Frames Ratio: r = {corr_matrix.loc['t_first_pressure_s', 'pressure_frames_ratio']:.2f}")
print(f"  - Line Height vs Width: r = {corr_matrix.loc['rm_line_height_median_x_mean', 'rm_width_mean_m']:.2f}")

## Summary

You've learned how to:
1. ✅ Normalize coordinates for cross-build-up comparison
2. ✅ Infer ball carrier from tracking data
3. ✅ Compute pressure metrics (intensity, timing, bursts)
4. ✅ Compute compactness metrics (width, length, hull area, line height)
5. ✅ Interpret feature values tactically
6. ✅ Batch-extract features for all build-ups
7. ✅ Analyze feature distributions and correlations

## Next Steps

- **Tutorial 3**: Train GMM zone models and NMF topic models to discover pressing patterns
- **Tutorial 4**: Create visualizations (heatmaps, networks, animations)

## Tactical Interpretation Guide

| Metric | Low | Medium | High |
|--------|-----|--------|------|
| `t_first_pressure_s` | <2s (Aggressive) | 2-4s (Controlled) | >4s (Reactive) |
| `pressure_frames_ratio` | <0.3 (Sporadic) | 0.3-0.5 (Moderate) | >0.5 (Sustained) |
| `rm_width_mean_m` | <20m (Compact) | 20-30m (Balanced) | >30m (Extended) |
| `rm_line_height_median_x_mean` | <-10m (Deep) | -10 to +10m (Mid) | >+10m (High) |